In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [2]:
iris = load_iris()
X, y = iris.data, iris.target

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3, 
    random_state=42
)

In [1]:
import optuna
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# =========================
# 1. Carregar dataset
# =========================
X, y = load_iris(return_X_y=True)

# Separar treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# =========================
# 2. Função objetivo do Optuna
# =========================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 2, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
    }

    model = RandomForestClassifier(
        **params,
        random_state=42,
        n_jobs=-1
    )

    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
    return scores.mean()

# =========================
# 3. Rodar Optuna
# =========================
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

# =========================
# 4. Pegar os melhores trials
# =========================
top_trials = sorted(study.trials, key=lambda t: t.value, reverse=True)[:3]

# =========================
# 5. Criar modelos com os melhores parâmetros
# =========================
models = []

for i, trial in enumerate(top_trials):
    params = trial.params
    
    rf = RandomForestClassifier(
        **params,
        random_state=42,
        n_jobs=-1
    )
    
    models.append((f"rf_{i}", rf))

# =========================
# 6. Criar VotingClassifier
# =========================
voting_clf = VotingClassifier(
    estimators=models,
    voting="soft"  # usa probabilidades (melhor na maioria dos casos)
)

# =========================
# 7. Treinar e avaliar
# =========================
voting_clf.fit(X_train, y_train)

y_pred = voting_clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Acurácia do VotingClassifier: {acc:.4f}")

/home/william/Projetos/mqtt_under_attack/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-03-17 20:36:54,645] A new study created in memory with name: no-name-1ef0dfa8-6c7b-4e5d-8c64-b21baaa8e087
[I 2026-03-17 20:36:55,556] Trial 0 finished with value: 0.9583333333333334 and parameters: {'n_estimators': 224, 'max_depth': 10, 'min_samples_split': 7, 'min_samples_leaf': 3, 'max_features': None}. Best is trial 0 with value: 0.9583333333333334.
[I 2026-03-17 20:36:56,639] Trial 1 finished with value: 0.9666666666666668 and parameters: {'n_estimators': 264, 'max_depth': 17, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.9666666666666668.
[I 2026-03-17 20:36:57,736] Trial 2 finished with value: 0.9583333333333334 and parameters: {'n_estimator

Acurácia do VotingClassifier: 0.9667
